# 🔦 TrustyAI
By testing our model after training and before it goes into production, we get a good understand on how well the model performs on data which we currently have available.  
Unfortunately, the real world is not very simple and things are constantly changing, so what we took for granted yesterday may look completely different today. Because of this, we need to monitor if changes in the world has an impact on our models ability to make accurate predictions.  
We want to track key metrics that can indicate if the model or the data starts behaving strangely once the model is in production and sees live data.  
For this, we have a tool called TrustyAI that can help us generate metrics for fairness and drift detection. With TrustyAI, we can determine if live data differs from what we expected when we created our training dataset. It allows us to measure whether our model responds to real-world data differently than we initially anticipated.

In [ ]:
!pip -q install onnxruntime model-registry

### Get your User token
We need to provide our user token so that the workbench can send request to our TrustyAI Service.
You can get the user token by:
1. Going to the OpenShift Console
2. Click the dropdown in the top right corner where your username is displayed
3. Choose "Copy login command"
4. Log in
5. Select the part in the first code box after `token`, it should look something like `sha256~.....`

### Get your Model Version

We are going to fetch some artifacts from the model training pipeline we ran earlier, and we will do it by utilizing the Model Registry that keeps track of what pipeline was ran.  
To do that, we need to point out what model version we are interested in.  
Go to the Model Registry called *userX*-prod-registry and get the **first** model version, which is the git hash looking thing.

### Configure Inference Endpoints and Model Settings

In [ ]:
# user token -> get it from the console or by 'oc whoami -t'
token = "sha256~XXXXXXXXXX"

# data information
data_bucket: str = "jukebox/artifacts"
data_version: int = 1
data_pointer: str = f"{data_bucket}/{data_version}"

# model metadata
model_version = "1"
# the model we want to do inference with
model_name = "jukebox-onnx-1-onnx"
model_namespace = "project"
cluster_domain = "DOMAIN.TLD"
infer_endpoint = f"https://{model_name}-{model_namespace}.{cluster_domain}"

# models api endpoint
api_endpoint = f"{infer_endpoint}/v2/models"

In [ ]:
# trusty ai service endpoints
namespace = "flan-t5-finetune"
trustyai_base_url = f"http://trustyai-service.{namespace}.svc.cluster.local"
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

In [ ]:
try:
    import os
    import sys
    import onnxruntime as rt
    import pickle
    import requests
    import torch
    import pandas as pd
    from pprint import pprint
except ImportError as e:
    print(f"{e}")

# local imports
parent_dir = os.path.abspath('..')
# the parent_dir could already be there if the kernel was not restarted, and we run this cell again
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

try:
    from libs.utilities import download_from_s3
    from libs.dataset import MergedDataset
except ImportError as e:
    print(f"{e}")

In [ ]:
# get api tokens to access to s3
aws_s3_endpoint: str = os.environ.get("AWS_S3_ENDPOINT", "")
access_key: str = os.environ.get("AWS_ACCESS_KEY_ID", "")
secret_key: str = os.environ.get("AWS_SECRET_ACCESS_KEY", "")

In [ ]:
# make sure model is deployed
pprint(requests.get(f"{api_endpoint}/{model_name}").json())

### Prepare TrainingData
We are going to send our training data to the TrustyAI Service so that it can compare the training data with the new data coming in from our inference requests.  
We limit it to just 5000 samples to keep it light, but in a real usecase you would send your full training data, or a part of it that properly represented its distribution.

In [ ]:
# prepare access credentials dictionary for s3 access
access_creds: dict = {
    "aws_access_key_id": access_key,
    "aws_secret_access_key": secret_key,
}

# we download training artifacts from the s3 registry
training_artifacts = {
    "training_data": f"{data_pointer}/train_dataset.parquet",
    "test_data": f"{data_pointer}/test_dataset.parquet",
    "scaler": f"{data_pointer}/minmax_scaler.pkl",
    "label_encoder": f"{data_pointer}/label_encoder.pkl",
    "model_checkpoint": "jukebox/onnx/1/country_predictor_model.onnx",
    "merged_data": f"{data_pointer}/merged_dataset_full.parquet",
}


In [ ]:
# download resources from s3
for k in training_artifacts.keys():
    obj_id = training_artifacts.get(k)
    print(f"Getting object: {obj_id}")
    download_from_s3(aws_s3_endpoint,
                     "models",
                     obj_id,
                     obj_id,
                     access_creds
                    )

In [ ]:
# features...
label_feature = 'country'
dataset_features = ['is_explicit', 'duration_ms', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']

# load datasets
song_dataset = MergedDataset(merged_dataset=training_artifacts.get("merged_data"),
                             features=dataset_features, label_column=label_feature)

# print dataset info
print(f"Dataset Contains {len(song_dataset)} items.")

# load encoder and scaler
with open(training_artifacts.get("scaler"), 'rb') as handle:
    scaler = pickle.load(handle)

with open(training_artifacts.get("label_encoder"), 'rb') as handle:
    label_encoder = pickle.load(handle)

# Country codes are strings. Here we are giving a unique number to each country
# so we can treat each country as number instead of strings
# because computers don't like strings
country_mapping = {i:c for i, c in enumerate(song_dataset.merged_dataset_full['country'].unique())}

In [ ]:
# select 5k samples
num_samples: int = 5000
training_samples = []
for i in range(num_samples):
    training_samples.append(song_dataset[i])

Data drift is calculated as Mean-Shift metric: meaning that for each test data we feed to the model, a metric is calculated that compares the new data to the training data set it knows.
This P-Value calculates the probability that the new data is coming from the same probability distribution as the training data.

P-Values nearing 1.0 means that the new data is still correctly described by training data, anything below 0.5 means that the model is showing significant data drift (e.g. training data is no more representative of the real phenomenon it is trying to predict)

In [ ]:
# data drift dataset setup

# 1- Load prediction model from disk
predictor = rt.InferenceSession(training_artifacts.get("model_checkpoint"),
                                providers=rt.get_available_providers())

# 2- get inputs and outputs of the onnx model
onnx_input = predictor.get_inputs()[0]
onnx_output = predictor.get_outputs()[0]
input_name = onnx_input.name
output_name = onnx_output.name

print(f"ONNX Model:\n Input: {input_name}, shape: {onnx_input.shape}\n Output: {output_name}, shape: {onnx_output.shape}")

In [ ]:
# 3- make a prediction using the model
y_pred_temp = []
y_test_temp = []
for point, label in training_samples:
    point = point.reshape(onnx_input.shape)
    y_pred_temp.append(predictor.run([output_name], {input_name: point.numpy()}))
    label = label.reshape(onnx_output.shape)
    y_test_temp.append(label.numpy())

After we have all our data, we structure it in a specific way that TrustyAI expects.  

The payload must contain:

1. the 'data_tag' field, describing the purpose of this dataset. (e.g. TRAINING)
2. the input name of the data point fed into the model (e.g the 'song_features')
3. the datatype (e.g. FP32)
4. the data shape for the input (e.g 5000 datapoints with 13 features -> [5000,13])
5. the data itself (e.g. a list of arrays)

Optionally, the results of the predictions can be attached to the dataset payload with these data:

1. the output name (e.g. "country_prediction")
2. the output data type (e.g. FP32)
3. the output shape (e.g. 5000 predictions over 73 counties -> [5000,73])
4. the prediction data itself (e.g. a list of arrays)

In [ ]:
# trusty ai data parameters
model_input_name: str = "song_features"
model_output_name: str = "country_predictions"

# input parms
input_shape = [num_samples, 13]
input_data = [i.numpy().tolist() for i, _ in training_samples]

# output parms
output_shape = [num_samples, 73]
output_data = [o[0][0].tolist() for o in y_pred_temp]

# prepare dataset payload in TrustyAI Format
trusty_ai_training_data = {
    "model_name": model_name,
    "data_tag": "TRAINING",
    "request": {
        "inputs": [
           {
                "name": model_input_name,
                "shape": input_shape,
                "datatype": "FP32",
                "data": input_data
            }
        ]
    },
    "response": {
        "model_name": model_name,
        "model_version": model_version,
        "outputs": [
            {
                "name": model_output_name,
                "datatype": "FP32",
                "shape": output_shape,
                "data": output_data
            }
        ]
    }
}

### Interacting with TrustyAI through requests
We will be interacting with our TrustyAI Service through rest requests.  
First thing we do is upload the data which we have prepared.

In [ ]:
# Upload data
endpoint = "data/upload"
url = f"{trustyai_base_url}/{endpoint}"
print(f"TrustyAI Service @ {url}")

# send training data to service
response = requests.post(url, headers=headers, json=trusty_ai_training_data)
print(response.text)

Now we can subscribe to our drift detection metric which will cause our TrustyAI Service to continously publish drift (specifically meanshift) metrics.  

The meanshift metric track how different the distribution of the training data looks like compared to the new data we send it.  

For each individual input and output feature we will get a "p-value" between 0 and 1. A p-value of 1.0 indicates a very high likelihood that the train and test data come from the same distribution, while a p-value < 0.05 indicates a statistically significant drift between train and test set.


In [ ]:
# Monitor meanshift
endpoint = "/metrics/drift/meanshift/request"
url = f"{trustyai_base_url}/{endpoint}"

payload = {
    "modelId": model_name,
    "referenceTag": "TRAINING"
}

response = requests.post(url, headers=headers, json=payload)
print(response.text)

To make sure all looks correct, we can ask the TrustyAI Service how our current setup looks like.  
We will get back information on what metrics we have subscribed to, as well as what data has been added.  

In [ ]:
# remap input fields to something more useful
input_map: dict = {}
for i in range(len(dataset_features)):
    input_map[f"{model_input_name}-{i}"] = f"{dataset_features[i]}"

output_map: dict = {}
for k in country_mapping.keys():
    output_map[f"{model_output_name}-{k}"] = f"{country_mapping[k]}"

feature_remap_dict = {
    "modelId": model_name,
    "inputMapping": input_map,
    "outputMapping": output_map,
}

print(feature_remap_dict)

# remap values...
endpoint = "/info/names"
url = f"{trustyai_base_url}/{endpoint}"
response = requests.post(url, headers=headers, json=feature_remap_dict)
print(response.text)

In [ ]:
# See what we have registered
endpoint = "/info"
url = f"{trustyai_base_url}/{endpoint}"
response = requests.get(url, headers=headers)
print(response.text)

### Send a request
Finally, we need to send a single request to our model server for TrustyAI to start publishing the metrics. This is so that it has at least one inference datapoint to compare its training data to.

⛵️ Introducing Drift

Now that we have set up TrustyAI to detect drifts, let's introduce some and see what happens.
To do this, we will simply send a bunch of requests based on our testing dataset.
Because we only sent a small size of our training dataset to TrustyAI, there is a good chance that the testing dataset will look significantly different.

In [ ]:
# make a prediction from the kserve inference endpoint
kserve_inference_url = f"{infer_endpoint}/v2/models/{model_name}/infer"

def rest_request(url, data, verify=False):
    json_data = {
        "inputs": [
            {
                "name": "input_sample",
                "shape": [1, 13],
                "datatype": "FP32",
                "data": data,
            }
        ]
    }

    response = requests.post(url, json=json_data, verify=verify)
    response_dict = response.json()
    return response_dict['outputs'][0]['data']

In [ ]:
# load a test dataset
test_samples: int = 500
test_dataset = pd.read_parquet(training_artifacts.get("test_data"))
test_samples = test_dataset[:test_samples]

# prepare data for inference & perform inference over test dataset
scaled_data = pd.DataFrame(scaler.transform(test_samples), columns=test_samples.columns)
for idx in range(len(scaled_data)):
    tensor_sample = torch.tensor(scaled_data.iloc[idx].values)
    prediction = rest_request(kserve_inference_url, tensor_sample.tolist(), verify=True)

Additionally, we can also monitor the average value of any feature to see how it changes over time.

In [ ]:
# Monitor the average value
endpoint = "/metrics/identity/request"
url = f"{trustyai_base_url}/{endpoint}"

payload = {
    "modelId": model_name,
    "columnName": "duration_ms",
    "batchSize": 256,
}

response = requests.post(url, headers=headers, json=payload)
print(response.text)

Let's go to the next Notebook to create some drift and observe the metrics in OpenShift UI